# Resumo de Testes Estatísticos

Referência rápida de testes de hipótese, correlação e planejamento amostral: fórmula, quando usar, e código pronto pra copiar/colar. Cada seção reimporta o que precisa, então dá pra pular direto pra seção que você precisa sem rodar o notebook inteiro.

## Guia de decisão rápida

| Situação | Tipo de dado | Teste recomendado |
|---|---|---|
| Comparar 1 amostra a um valor de referência | Média contínua | Teste t de 1 amostra (seção 5) |
| Comparar 1 proporção a um valor de referência | Binário | Teste Z de 1 proporção (seção 5) |
| Comparar médias de 2 grupos independentes (dados ~normais) | Contínuo | Teste t independente — Welch se variâncias diferentes (seções 6–7) |
| Comparar 2 grupos independentes (não normais / outliers) | Contínuo | Mann-Whitney U (seção 13) |
| Comparar 2 medições na mesma unidade (antes/depois) | Contínuo | Teste t pareado (seção 8) |
| Comparar 2 proporções independentes (ex: conversão A/B) | Binário | Teste Z de duas proporções (seções 1, 3, 4) |
| Comparar médias de 3+ grupos | Contínuo | ANOVA + Tukey HSD se significativo (seções 10–11) |
| Testar associação entre 2 variáveis categóricas | Categórico | Qui-quadrado — Fisher exato se amostra pequena (seção 9) |
| Medir associação entre 2 variáveis contínuas | Contínuo | Pearson (linear) ou Spearman (monotônica/outliers) (seções 14–15) |
| Planejar `n` antes de coletar dados | – | Cálculo de tamanho de amostra / poder estatístico (seções 16–19) |
| Rodando vários testes ao mesmo tempo | – | Correção de múltiplos testes (seção 12) |

**Convenção usada no notebook inteiro:** `alpha = 0.05`, testes bicaudais, a menos que indicado o contrário.


## 1. Teste Z para diferença entre duas proporções (A/B test)

**O que é:** testa se a diferença entre duas taxas de conversão (proporções) é estatisticamente significativa, ou se pode ser só ruído amostral.

**Quando usar:** dados binários (converteu / não converteu) em dois grupos independentes (A e B), com amostra grande o suficiente pra aproximação normal (regra prática: `n*p` e `n*(1-p)` ≥ 5 em cada grupo).

**Hipóteses:**
- H0: `p_A = p_B` (não há diferença real de conversão)
- H1: `p_A ≠ p_B` (teste bicaudal, padrão do `proportions_ztest`)

**Estatística:**

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}$$

onde `p̂` é a proporção combinada (pooled) das duas amostras.

**Código abaixo:** calcula conversão de A e B a partir de vetores binários, a diferença absoluta, o lift relativo, e roda o teste z retornando o p-valor.

In [1]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Dados (amostra maior: 200 visitantes por grupo, gerados por simulacao)
np.random.seed(42)
A = np.random.binomial(1, 0.30, 200)  # grupo A: conversao real ~30%
B = np.random.binomial(1, 0.50, 200)  # grupo B: conversao real ~50%

# Conversion rates
conversion_A = A.mean()
conversion_B = B.mean()

# Difference
difference = conversion_B - conversion_A

# Relative lift
lift = difference / conversion_A

# Hypothesis test
successes = np.array([A.sum(), B.sum()])
nobs = np.array([len(A), len(B)])

z_stat, p_value = proportions_ztest(
    successes,
    nobs
)

print("Conversion A:", conversion_A)
print("Conversion B:", conversion_B)
print("Difference:", difference)
print("Lift:", lift)
print("p-value:", p_value)

Conversion A: 0.3
Conversion B: 0.545
Difference: 0.24500000000000005
Lift: 0.8166666666666669
p-value: 7.051366865845907e-07


## 2. Decisão estatística (regra do p-valor)

**O que é:** compara o p-valor obtido no teste anterior com o nível de significância `alpha` (aqui 5%) pra decidir se rejeita ou não H0.

**Regra:**
- `p-value < alpha` → rejeita H0 → diferença estatisticamente significativa
- `p-value ≥ alpha` → não rejeita H0 → não há evidência suficiente de diferença

**Cuidado de interpretação:** "não significativo" não prova que as proporções são iguais, só que a amostra não teve poder suficiente pra detectar a diferença com confiança.

**Código abaixo:** aplica essa regra ao `p_value` calculado na célula anterior.

In [2]:
alpha = 0.05

if p_value < alpha:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Statistically significant difference


## 3. Teste Z de proporções a partir de contagens agregadas

**O que é:** mesmo teste do bloco 1 (teste z para duas proporções), mas agora os dados já vêm agregados como contagem de sucessos e total de observações por grupo, em vez de vetores binários individuais — útil quando você só tem os totais (ex: relatório de campanha).

**Hipóteses:** as mesmas do teste anterior (H0: proporções iguais vs H1: proporções diferentes).

**Interpretação do resultado:** aqui `z` é negativo porque a segunda proporção (70/1000 = 7%) é maior que a primeira (50/1000 = 5%), então a diferença tem sinal invertido. O p-valor (~0,06) fica levemente acima do `alpha = 0,05` comum, ou seja, no limiar da significância.

**Código abaixo:** roda `proportions_ztest` diretamente com listas de sucessos e observações.

In [3]:
from statsmodels.stats.proportion import proportions_ztest

sucessos = [50, 70]
observacoes = [1000, 1000]

z, p = proportions_ztest(sucessos, observacoes)

print(z)
print(p)

-1.883108942886774
0.059685605532426224


## 4. Intervalo de confiança para diferença de duas proporções

**O que é:** complementa o teste Z de proporções (seções 1 e 3) — em vez de só dizer "significativo ou não", estima a faixa provável do tamanho real da diferença entre as duas proporções, com um nível de confiança escolhido.

**Por que isso importa na prática:** um IC comunica mais do que um p-valor sozinho. "A diferença está entre 1 e 3 pontos percentuais, com 95% de confiança" costuma ser mais acionável pra decisão de negócio do que só "p<0,05". Repare também que, se o IC cruza o zero, isso é consistente com um resultado "não significativo" — as duas leituras devem bater.

**Fórmula (Wald):**

$$ (\hat p_B - \hat p_A) \pm Z \cdot \sqrt{\frac{\hat p_A(1-\hat p_A)}{n_A} + \frac{\hat p_B(1-\hat p_B)}{n_B}} $$

**Código abaixo:** usa as mesmas contagens agregadas da seção 3 (50/1000 vs 70/1000) pra calcular o IC de 95% da diferença.


In [4]:
import numpy as np
from scipy.stats import norm

sucessos = np.array([50, 70])
observacoes = np.array([1000, 1000])

p_A, p_B = sucessos / observacoes
diff = p_B - p_A

confidence = 0.95
Z = norm.ppf((1 + confidence) / 2)

se = np.sqrt(p_A*(1-p_A)/observacoes[0] + p_B*(1-p_B)/observacoes[1])
ic_low, ic_high = diff - Z*se, diff + Z*se

print(f"Diferença (B - A): {diff:.4f}")
print(f"IC {confidence:.0%}: [{ic_low:.4f}, {ic_high:.4f}]")


Diferença (B - A): 0.0200
IC 95%: [-0.0008, 0.0408]


## 5. Teste para uma amostra (comparação com valor de referência)

**O que é:** diferente dos testes anteriores (que comparam dois grupos), aqui você compara **uma única amostra** contra um valor fixo/hipotético — um benchmark, uma meta, um valor histórico.

**Quando usar:**
- **Teste t de 1 amostra:** variável contínua. Ex: "o tempo médio de atendimento da equipe é diferente da meta de 10 minutos?"
- **Teste Z de 1 proporção:** variável binária. Ex: "nossa taxa de conversão atual é diferente do benchmark de mercado de 20%?"

**Hipóteses (caso da média):**
- H0: `média = valor_de_referência`
- H1: `média ≠ valor_de_referência`

**Código abaixo:** `ttest_1samp` compara uma amostra de tempos de atendimento contra a meta de 10 minutos; `proportions_ztest` com `value=` compara uma proporção observada contra o benchmark de 20%.


In [5]:
from scipy.stats import ttest_1samp
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# --- Teste t de 1 amostra ---
np.random.seed(11)
tempos_atendimento = np.random.normal(11.2, 2.5, 40).round(1)  # minutos
meta = 10.0

t, p_t = ttest_1samp(tempos_atendimento, popmean=meta)
print("Teste t de 1 amostra")
print("Média observada:", tempos_atendimento.mean().round(2))
print("p-value:", p_t)

# --- Teste Z de 1 proporção ---
conversoes = 46
visitantes = 180
benchmark = 0.20

z, p_z = proportions_ztest(conversoes, visitantes, value=benchmark)
print("\nTeste Z de 1 proporção")
print("Proporção observada:", round(conversoes/visitantes, 3))
print("z:", z)
print("p-value:", p_z)


Teste t de 1 amostra
Média observada: 11.18
p-value: 0.002470460216418061

Teste Z de 1 proporção
Proporção observada: 0.256
z: 1.7088539142345303
p-value: 0.08747801300374133


## 6. Checagem de premissas: normalidade e homogeneidade de variância

**O que é:** antes de escolher entre teste paramétrico (t-test, ANOVA) e não-paramétrico (Mann-Whitney), ou entre t de Student e t de Welch, é preciso checar as premissas que os textos das seções seguintes mencionam mas não testam explicitamente.

**Testes usados:**
- **Shapiro-Wilk** (`shapiro`): testa se uma amostra vem de uma distribuição normal. H0 = os dados são normais. `p < alpha` → evidência **contra** a normalidade.
- **Levene** (`levene`): testa se dois ou mais grupos têm variâncias iguais (homocedasticidade). H0 = variâncias iguais. `p < alpha` → variâncias **diferentes** → use `equal_var=False` (Welch) no t-test.

**Regra prática:** com amostras grandes (n > 30), o Teorema Central do Limite já dá alguma robustez ao t-test mesmo com não-normalidade leve — o Shapiro fica mais crítico em amostras pequenas.

**Código abaixo:** aplica os dois testes aos mesmos grupos A e B usados no teste t independente (seção 7).


In [6]:
from scipy.stats import shapiro, levene
import numpy as np

np.random.seed(1)
A = np.random.normal(90, 15, 30).round(1)
B = np.random.normal(105, 15, 30).round(1)

shapiro_A = shapiro(A)
shapiro_B = shapiro(B)
levene_result = levene(A, B)

print("Shapiro-Wilk A: statistic=%.4f, p=%.4f" % shapiro_A)
print("Shapiro-Wilk B: statistic=%.4f, p=%.4f" % shapiro_B)
print("Levene (variâncias): statistic=%.4f, p=%.4f" % levene_result)

alpha = 0.05
normal_ok = shapiro_A.pvalue > alpha and shapiro_B.pvalue > alpha
var_ok = levene_result.pvalue > alpha

print(f"\nNormalidade OK em ambos os grupos: {normal_ok}")
print(f"Variâncias iguais (usar Student, não Welch): {var_ok}")


Shapiro-Wilk A: statistic=0.9735, p=0.6379
Shapiro-Wilk B: statistic=0.9484, p=0.1528
Levene (variâncias): statistic=1.0933, p=0.3001

Normalidade OK em ambos os grupos: True
Variâncias iguais (usar Student, não Welch): True


## 4. Teste t de Student para amostras independentes

**O que é:** compara as médias de dois grupos **independentes** (sujeitos diferentes em A e B) pra ver se a diferença entre elas é estatisticamente significativa.

**Quando usar:** variável contínua, dois grupos não pareados, aproximadamente normal (ou `n` grande o suficiente pelo Teorema Central do Limite). Por padrão o `ttest_ind` do scipy assume variâncias iguais entre os grupos (teste t de Student clássico); se as variâncias forem muito diferentes, o certo é usar `equal_var=False` (teste de Welch).

**Hipóteses:**
- H0: `média_A = média_B`
- H1: `média_A ≠ média_B`

**Código abaixo:** compara os grupos A e B (ex: dois métodos, dois tratamentos) e retorna o p-valor da diferença de médias.
**Novidade nesta versão:** além do p-valor, o código agora reporta o **Cohen's d** (tamanho do efeito — 0,2 pequeno / 0,5 médio / 0,8 grande, regra de Cohen) e o **intervalo de confiança de 95%** da diferença de médias. O p-valor sozinho não diz o quão grande é a diferença; o `equal_var` usado aqui deve ser decidido com base na checagem de premissas da seção 6.


In [7]:
from scipy.stats import ttest_ind
import numpy as np

# Amostra maior: 30 observacoes por grupo
np.random.seed(1)
A = np.random.normal(90, 15, 30).round(1)
B = np.random.normal(105, 15, 30).round(1)

result = ttest_ind(A, B)  # equal_var=True por padrão — ver checagem de premissas (seção 6)
t, p = result.statistic, result.pvalue

# Effect size (Cohen's d, desvio-padrão combinado)
n_A, n_B = len(A), len(B)
pooled_std = np.sqrt(((n_A-1)*A.var(ddof=1) + (n_B-1)*B.var(ddof=1)) / (n_A+n_B-2))
cohens_d = (B.mean() - A.mean()) / pooled_std

# Intervalo de confiança de 95% para a diferença de médias
ci_low, ci_high = result.confidence_interval(confidence_level=0.95)

print("p-value:", p)
print("Cohen's d:", round(cohens_d, 3))
print(f"IC 95% da diferença (B - A): [{ci_low:.2f}, {ci_high:.2f}]")


p-value: 1.7493992371161784e-05
Cohen's d: 1.209
IC 95% da diferença (B - A): [-24.33, -9.76]


## 5. Teste t pareado (amostras dependentes)

**O que é:** compara duas medições feitas **no mesmo indivíduo/unidade**, em dois momentos (antes/depois) ou duas condições — diferente do teste independente, aqui cada linha de "antes" tem seu par exato em "depois".

**Por que usar pareado em vez de independente:** ao comparar a mesma unidade contra ela mesma, você remove a variação individual (cada pessoa é seu próprio controle), o que aumenta o poder do teste pra detectar diferenças reais.

**Hipóteses:**
- H0: a diferença média entre pares é zero (`média(depois - antes) = 0`)
- H1: a diferença média entre pares é diferente de zero

**Código abaixo:** compara `antes` vs `depois` (ex: performance pré/pós intervenção) usando `ttest_rel`.
**Novidade nesta versão:** adicionado o **Cohen's dz**, a versão do tamanho do efeito específica pra dados pareados — usa o desvio-padrão das *diferenças* (depois − antes), não o desvio-padrão dos grupos brutos, porque aqui a unidade de análise é o par, não o indivíduo isolado.


In [8]:
from scipy.stats import ttest_rel
import numpy as np

# Amostra maior: 25 pares antes/depois
np.random.seed(7)
antes = np.random.normal(11, 3, 25).round(1)
melhora = np.random.normal(2, 1.5, 25).round(1)
depois = (antes + melhora).round(1)

t, p = ttest_rel(antes, depois)

diffs = depois - antes
cohens_dz = diffs.mean() / diffs.std(ddof=1)

print("p-value:", p)
print("Cohen's dz:", round(cohens_dz, 3))


p-value: 0.0001869487103792182
Cohen's dz: 0.882


## 6. Teste Qui-quadrado de independência

**O que é:** testa se existe associação entre **duas variáveis categóricas**, organizadas em uma tabela de contingência (linhas x colunas de contagens).

**Quando usar:** dados de contagem/frequência (não médias), ex: "grupo (controle/tratamento) x converteu (sim/não)".

**Hipóteses:**
- H0: as variáveis são independentes (não há associação)
- H1: as variáveis são dependentes (há associação)

**Como funciona:** compara as frequências observadas com as frequências esperadas caso as variáveis fossem independentes; quanto maior a distância entre observado e esperado, maior a estatística qui-quadrado e menor o p-valor.

**Código abaixo:** roda `chi2_contingency` numa tabela 2x2 de conversões (ex: grupo x converteu/não converteu).
**Cuidado com o pressuposto:** o qui-quadrado assume frequência esperada ≥ 5 em cada célula da tabela. Com contagens pequenas (amostra pequena ou categoria rara) esse pressuposto quebra e o p-valor fica pouco confiável — nesse caso, use o **teste exato de Fisher** (`fisher_exact`), que não depende dessa aproximação e é exato mesmo em amostras pequenas (tabelas 2x2).


In [9]:
from scipy.stats import chi2_contingency

tabela = [
    [120, 880],
    [150, 850]
]

chi2, p, dof, expected = chi2_contingency(tabela)

print(p)

0.057746841509689


In [10]:
from scipy.stats import fisher_exact

# Amostra pequena (frequências esperadas < 5) -> qui-quadrado não é confiável aqui
tabela_pequena = [
    [2, 8],
    [6, 4]
]

odds_ratio, p_fisher = fisher_exact(tabela_pequena)
print("Fisher exato — odds ratio:", odds_ratio)
print("Fisher exato — p-value:", p_fisher)


Fisher exato — odds ratio: 0.16666666666666666
Fisher exato — p-value: 0.16980233388902116


## 7. ANOVA de um fator (F-test)

**O que é:** compara as médias de **três ou mais grupos** simultaneamente, testando se pelo menos um deles difere dos demais.

**Por que não usar vários t-tests em pares:** cada teste par a par tem chance de erro Tipo I (falso positivo); ao rodar vários testes o erro acumulado (inflação do alpha) cresce. A ANOVA testa todos os grupos de uma vez, controlando esse problema.

**Hipóteses:**
- H0: todas as médias são iguais (`média_A = média_B = média_C`)
- H1: pelo menos uma média é diferente das outras

**Limitação:** a ANOVA diz que existe diferença, mas não diz **qual** grupo difere — pra isso é necessário um teste post-hoc (ex: Tukey HSD, ver seção 11).

**Código abaixo:** compara três grupos (A, B, C) com `f_oneway` e retorna o p-valor do teste F.
**Novidade nesta versão:** adicionado o **eta² (η²)**, o tamanho do efeito da ANOVA — proporção da variância total explicada pela diferença entre grupos (0,01 pequeno / 0,06 médio / 0,14 grande, regra de Cohen).


In [11]:
from scipy.stats import f_oneway
import numpy as np

# Amostra maior: 30 observacoes por grupo
np.random.seed(3)
A = np.random.normal(88, 8, 30).round(1)
B = np.random.normal(104, 8, 30).round(1)
C = np.random.normal(76, 8, 30).round(1)

F, p = f_oneway(A, B, C)

grand_mean = np.concatenate([A, B, C]).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in (A, B, C))
ss_total = sum(((g - grand_mean)**2).sum() for g in (A, B, C))
eta_squared = ss_between / ss_total

print("p-value:", p)
print("Eta²:", round(eta_squared, 3))


p-value: 4.555858799454035e-17
Eta²: 0.579


## 11. Teste post-hoc de Tukey HSD

**O que é:** a ANOVA (seção 10) só diz que existe diferença entre os grupos, sem apontar qual par difere. O Tukey HSD faz todas as comparações par a par (A vs B, A vs C, B vs C) já com a correção pra múltiplos testes embutida, controlando o erro tipo I acumulado — é um caso específico do problema tratado de forma mais geral na seção 12.

**Quando usar:** depois de uma ANOVA significativa, quando é preciso saber especificamente quais grupos diferem entre si.

**Código abaixo:** usa os mesmos grupos A, B, C da ANOVA e retorna uma tabela com a diferença de médias, o p-valor ajustado e se rejeita H0 pra cada par.


In [12]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np

np.random.seed(3)
A = np.random.normal(88, 8, 30).round(1)
B = np.random.normal(104, 8, 30).round(1)
C = np.random.normal(76, 8, 30).round(1)

valores = np.concatenate([A, B, C])
grupos = ['A']*len(A) + ['B']*len(B) + ['C']*len(C)

tukey = pairwise_tukeyhsd(endog=valores, groups=grupos, alpha=0.05)
print(tukey)


 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj   lower    upper   reject
------------------------------------------------------
     A      B  14.3233    0.0   8.9825  19.6642   True
     A      C   -10.05 0.0001 -15.3908  -4.7092   True
     B      C -24.3733    0.0 -29.7142 -19.0325   True
------------------------------------------------------


## 12. Correção para múltiplos testes (Bonferroni / FDR)

**O que é:** o Tukey HSD (seção 11) já corrige múltiplas comparações automaticamente, mas isso vale só pra comparações par a par dentro de uma ANOVA. Quando você roda vários testes de hipótese **independentes** (ex: testar 5 métricas diferentes no mesmo experimento A/B), cada teste carrega 5% de chance de falso positivo — juntos, a chance de pelo menos um falso positivo entre os 5 cresce bem acima de 5%.

**Métodos:**
- **Bonferroni:** divide `alpha` pelo número de testes (`alpha/n`). Simples e conservador — controla bem o erro, mas perde poder estatístico.
- **FDR / Benjamini-Hochberg:** controla a proporção esperada de falsos positivos entre os testes rejeitados, em vez do erro por teste. Menos conservador que Bonferroni; mais comum quando se testa muitas métricas ao mesmo tempo (ex: dezenas de KPIs num experimento).

**Código abaixo:** aplica as duas correções a uma lista de 5 p-valores (simulando 5 métricas testadas no mesmo experimento).


In [13]:
from statsmodels.stats.multitest import multipletests

p_valores = [0.01, 0.04, 0.03, 0.20, 0.045]

rejeita_bonf, p_bonf, _, _ = multipletests(p_valores, alpha=0.05, method='bonferroni')
rejeita_fdr, p_fdr, _, _ = multipletests(p_valores, alpha=0.05, method='fdr_bh')

print("p-valores originais:  ", p_valores)
print("Bonferroni — rejeita H0:", rejeita_bonf.tolist())
print("Bonferroni — p ajustado:", p_bonf.round(4).tolist())
print("FDR (BH)   — rejeita H0:", rejeita_fdr.tolist())
print("FDR (BH)   — p ajustado:", p_fdr.round(4).tolist())


p-valores originais:   [0.01, 0.04, 0.03, 0.2, 0.045]
Bonferroni — rejeita H0: [True, False, False, False, False]
Bonferroni — p ajustado: [0.05, 0.2, 0.15, 1.0, 0.225]
FDR (BH)   — rejeita H0: [True, False, False, False, False]
FDR (BH)   — p ajustado: [0.05, 0.0562, 0.0562, 0.2, 0.0562]


## 8. Teste de Mann-Whitney U (não paramétrico)

**O que é:** equivalente não-paramétrico do teste t independente — compara se as distribuições de dois grupos independentes são diferentes, sem assumir normalidade dos dados.

**Quando usar:** quando os dados não são normais, quando a amostra é pequena, ou quando há outliers fortes que distorceriam a média (repare que o grupo A tem um valor de 25000 destoante dos demais — a média seria fortemente puxada por ele, mas o teste baseado em **ranks/medianas** é robusto a isso).

**Hipóteses:**
- H0: as duas distribuições são iguais (nenhuma tende a ter valores maiores que a outra)
- H1: uma distribuição tende a ter valores sistematicamente maiores que a outra

**Como funciona:** ao invés de comparar médias, o teste converte os valores em postos (ranks) combinados dos dois grupos e compara a soma dos ranks entre eles.

**Código abaixo:** compara os grupos A e B (com outlier em A) usando `mannwhitneyu`.

In [14]:
from scipy.stats import mannwhitneyu
import numpy as np

# Amostra maior: 30 observacoes por grupo (mantendo um outlier no grupo A)
np.random.seed(5)
A = np.random.normal(3500, 300, 29).round(0).tolist() + [25000]  # outlier
B = np.random.normal(3600, 300, 30).round(0).tolist()

u, p = mannwhitneyu(A, B)

print(p)

0.010092817132935963


## 9. Correlação de Pearson

**O que é:** mede a força e a direção da associação **linear** entre duas variáveis contínuas (ex: horas de estudo x nota).

**Quando usar:** ambas variáveis contínuas, relação esperada aproximadamente linear (não captura relações não-lineares).

**Interpretação do coeficiente `r`:**
- `r` varia de -1 a +1
- próximo de +1 → forte correlação positiva (uma sobe, a outra sobe)
- próximo de -1 → forte correlação negativa
- próximo de 0 → pouca ou nenhuma correlação linear

**Hipóteses do p-valor associado:**
- H0: `r = 0` (não há correlação linear na população)
- H1: `r ≠ 0`

**Cuidado:** correlação não implica causalidade — `r` alto só indica associação linear, não que uma variável causa a outra.

**Código abaixo:** calcula `r` e o p-valor entre `horas` de estudo e `notas` com `pearsonr`.

In [15]:
from scipy.stats import pearsonr
import numpy as np

# Amostra maior: 20 observacoes
np.random.seed(9)
horas = np.arange(1, 21)
notas = (50 + 2.0*horas + np.random.normal(0, 5, 20)).round(1)

r, p = pearsonr(horas, notas)

print(r)
print(p)

0.9614001751341554
1.5702616196622007e-11


## 15. Correlação de Spearman

**O que é:** versão não-paramétrica da correlação — mede associação **monotônica** entre duas variáveis (não precisa ser linear), usando os postos (ranks) dos valores em vez dos valores brutos.

**Quando usar:** quando a relação não é claramente linear, quando há outliers fortes que distorceriam o Pearson, ou quando os dados são ordinais.

**Interpretação:** mesma escala do Pearson (-1 a +1), mesma leitura de força/direção — só muda o que está sendo medido (ranks, não valores brutos).

**Código abaixo:** repete o exemplo de horas de estudo x notas da seção 14, mas acrescenta um aluno que estudou mais que todo mundo e tirou uma nota muito acima da tendência (um outlier de alavancagem — não quebra a ordem geral, só exagera a magnitude). Repare que o Pearson despenca (é sensível ao valor bruto do ponto extremo), enquanto o Spearman quase não muda (só olha a posição no ranking, e esse aluno segue sendo o 1º em horas e o 1º em nota).


In [16]:
from scipy.stats import pearsonr, spearmanr
import numpy as np

np.random.seed(9)
horas = np.arange(1, 21)
notas = (50 + 2.0*horas + np.random.normal(0, 5, 20)).round(1)

print("Sem outlier:")
r_p0, _ = pearsonr(horas, notas)
r_s0, _ = spearmanr(horas, notas)
print(f"Pearson r:  {r_p0:.3f}")
print(f"Spearman r: {r_s0:.3f}")

# Outlier de alavancagem: mais horas que todo mundo, nota muito acima da tendência
# (ainda é o maior valor em horas E em notas, então não inverte nenhum rank)
horas_out = np.append(horas, 21)
notas_out = np.append(notas, 300)

r_pearson, p_pearson = pearsonr(horas_out, notas_out)
r_spearman, p_spearman = spearmanr(horas_out, notas_out)

print("\nCom outlier:")
print(f"Pearson r:  {r_pearson:.3f} (p={p_pearson:.4f})")
print(f"Spearman r: {r_spearman:.3f} (p={p_spearman:.4f})")


Sem outlier:
Pearson r:  0.961
Spearman r: 0.950

Com outlier:
Pearson r:  0.603 (p=0.0038)
Spearman r: 0.957 (p=0.0000)


## 16. Tamanho de amostra para estimar uma média (margem de erro conhecida)

**O que é:** calcula quantas observações são necessárias pra estimar uma média populacional dentro de uma margem de erro desejada, com um nível de confiança escolhido — pensado **antes** de coletar os dados (planejamento amostral), diferente dos testes anteriores que analisavam dados já coletados.

**Quando usar:** quando você sabe (ou tem uma estimativa de) o desvio-padrão da variável (`sigma`), e quer definir de antemão o `n` mínimo do estudo/experimento.

**Fórmula:**

$$n = \left(\frac{Z \cdot \sigma}{E}\right)^2$$

onde `Z` é o valor crítico da normal padrão para o nível de confiança (ex: 1,96 para 95%), `σ` é o desvio-padrão estimado, e `E` é a margem de erro máxima que você aceita.

**Código abaixo:** com 95% de confiança, `sigma=10` e margem de erro de 2, calcula o `n` mínimo necessário.

In [17]:
from scipy.stats import norm
import math

# Parâmetros
confidence = 0.95
sigma = 10
margin_error = 2

# Valor crítico Z
Z = norm.ppf((1 + confidence) / 2)

# Cálculo
n = (Z * sigma / margin_error) ** 2

print(f"Valor de Z: {Z:.3f}")
print(f"Tamanho da amostra: {math.ceil(n)}")

Valor de Z: 1.960
Tamanho da amostra: 97


## 17. A mesma fórmula, encapsulada em função reutilizável

**O que é:** reaproveita a fórmula da célula anterior, agora dentro de uma função `sample_size_mean(confidence, sigma, margin_error)` — útil pra recalcular o tamanho de amostra rapidamente com parâmetros diferentes, sem duplicar código.

**Por que isso importa na prática:** em um projeto real você normalmente testa vários cenários (margens de erro ou desvios-padrão diferentes); ter isso como função evita copiar e colar a fórmula toda vez.

**Nota:** os parâmetros usados (`confidence=0.95, sigma=10, margin_error=2`) são os mesmos da célula anterior, por isso o resultado é idêntico (n=97).

**Código abaixo:** define a função e chama com os mesmos valores de antes, só que agora de forma reutilizável.

In [18]:
from scipy.stats import norm
import math

def sample_size_mean(confidence, sigma, margin_error):
    Z = norm.ppf((1 + confidence) / 2)
    n = (Z * sigma / margin_error) ** 2
    return math.ceil(n)

sample_size_mean(0.95, 10, 2)

97

## 18. Tamanho de amostra para estimar uma proporção

**O que é:** versão da fórmula de planejamento amostral para quando a variável de interesse é uma **proporção** (ex: taxa de conversão, taxa de aprovação, taxa de churn) em vez de uma média contínua.

**Fórmula:**

$$n = \frac{Z^2 \cdot p(1-p)}{E^2}$$

onde `p` é a proporção esperada (quando não se sabe, usa-se `p=0.5`, que maximiza `p(1-p)` e dá o tamanho de amostra mais conservador/seguro) e `E` é a margem de erro desejada.

**Quando usar:** planejar pesquisas de opinião, dimensionar quantos usuários observar pra estimar uma taxa de conversão com certa precisão, etc.

**Código abaixo:** com 95% de confiança, `p=0.40` esperado e margem de erro de 5 pontos percentuais, calcula o `n` mínimo necessário (369).

In [19]:
from scipy.stats import norm
import math

confidence = 0.95
p = 0.40
margin_error = 0.05

Z = norm.ppf((1 + confidence) / 2)

n = (Z**2 * p * (1 - p)) / (margin_error**2)

print(f"Tamanho da amostra: {math.ceil(n)}")

Tamanho da amostra: 369


## 19. Poder estatístico e tamanho de amostra para um teste A/B

**O que é:** diferente das duas células anteriores (que estimam uma proporção/média isolada), aqui calculamos quantas observações **por grupo** são necessárias pra conseguir **detectar** uma diferença específica entre duas proporções, dado um poder estatístico desejado — ou seja, planejamento de experimento (A/B test) antes de rodar.

**Conceitos-chave:**
- **Effect size (Cohen's h):** mede o "tamanho" da diferença entre as duas proporções numa escala padronizada, calculado aqui por `proportion_effectsize(p1, p2)`.
- **Power (poder do teste) = 1 − β:** probabilidade de detectar a diferença quando ela realmente existe (evitar o erro Tipo II — não detectar um efeito real). Aqui usa-se o padrão de mercado de 80%.
- **Alpha:** taxa de erro Tipo I aceita (falso positivo), aqui 5%.

**Cenário do código:** comparar uma conversão de 10% (p1) contra 12% (p2) — uma diferença pequena — com poder de 80% e alpha de 5%.

**Por que o `n` sai tão grande (≈3835 por grupo):** diferenças pequenas entre proporções exigem amostras muito maiores pra serem detectadas com confiança — conecta diretamente com o teste Z de proporções do início do notebook: com pouca amostra (como os primeiros exemplos), mesmo diferenças reais podem não aparecer como "significativas".

In [20]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Taxas de conversão
p1 = 0.10
p2 = 0.12

# Calcula o tamanho do efeito (Cohen's h)
effect_size = proportion_effectsize(p1, p2)

analysis = NormalIndPower()

n = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    ratio=1.0
)

print(f"Amostra por grupo: {n:.0f}")

Amostra por grupo: 3835
